In [1]:
import os
import sys

# Elimina las variables de entorno conflictivas ANTES de importar PySpark
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]
if "PYSPARK_SUBMIT_ARGS" in os.environ:
    del os.environ["PYSPARK_SUBMIT_ARGS"]

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round, count

# Configura la sesión de Spark utilizando los binarios internos
spark = SparkSession.builder \
    .appName("EBAC_Practica_Spark_Patrick") \
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print(f"Sesión de Spark iniciada. Versión: {spark.version}")

Sesión de Spark iniciada. Versión: 3.5.0


In [2]:
# Carga el archivo CSV a una estructura DataFrame de Spark
ruta_archivo = os.path.join("C:\\Users\\patri\\Downloads", "kc_house_data (1).csv")
df = spark.read.csv(ruta_archivo, header=True, inferSchema=True)

# Diagnostica la estructura y muestra las primeras filas
print("--- Esquema de Datos ---")
df.printSchema()
df.show(5)

--- Esquema de Datos ---
root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)

+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+--------

In [3]:
# Limpia los datos: elimina nulos y filtra valores en cero para habitaciones, baños o precio
df_clean = df.na.drop() \
    .filter((col("bedrooms") > 0) & (col("bathrooms") > 0) & (col("price") > 0))

print(f"Total de registros tras la limpieza: {df_clean.count()}")

Total de registros tras la limpieza: 21597


In [4]:
# Ordena el listado completo de columnas por zipcode
df_sorted = df_clean.orderBy(col("zipcode"))

print("--- Listado completo ordenado por zipcode ---")
df_sorted.show(5)

--- Listado completo ordenado por zipcode ---
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|   price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|4014400292|20150114T000000|465000.0|       3|      2.5|       2714|   17936|   2.0|         0|   0|        3|    9|      2714|            0|    2005|           0|  98001|47.3185|-122.275|         2590|     18386|
|9262800171|20150324T000000|252000.0|       4|      1.5|       1550|   19800|   1.0|         0|   

In [5]:
# Encuentra el zipcode con mayor número de casas
top_zip_row = df_clean.groupBy("zipcode").agg(count("*").alias("total_casas")).orderBy(col("total_casas").desc()).first()
top_zip = top_zip_row["zipcode"]
print(f"Zipcode con mayor número de casas: {top_zip}")

# Filtra por el zipcode seleccionado y calcula el promedio de precio y tamaño en m2
df_top = df_clean.filter(col("zipcode") == top_zip)
df_res = df_top.agg(
    round(avg("price"), 2).alias("precio_promedio"),
    round(avg("sqft_living") * 0.092903, 2).alias("tamaño_m2_promedio")
)

print(f"--- Promedios calculados para el zipcode {top_zip} ---")
df_res.show()

Zipcode con mayor número de casas: 98103
--- Promedios calculados para el zipcode 98103 ---
+---------------+------------------+
|precio_promedio|tamaño_m2_promedio|
+---------------+------------------+
|      584919.21|            153.37|
+---------------+------------------+



In [6]:
# Agrupa por zipcode, número de habitaciones y baños para obtener el precio promedio
df_grouped = df_clean.groupBy("zipcode", "bedrooms", "bathrooms").agg(
    round(avg("price"), 2).alias("precio_promedio")
).orderBy("zipcode", "bedrooms", "bathrooms")

# Muestra el resultado final del agrupamiento
print("--- Agrupamiento por zipcode, habitaciones y baños ---")
df_grouped.show(10)

--- Agrupamiento por zipcode, habitaciones y baños ---
+-------+--------+---------+---------------+
|zipcode|bedrooms|bathrooms|precio_promedio|
+-------+--------+---------+---------------+
|  98001|       1|      1.0|       166000.0|
|  98001|       1|      2.0|       171000.0|
|  98001|       2|      1.0|      197428.57|
|  98001|       2|      1.5|       350000.0|
|  98001|       2|     1.75|       246112.5|
|  98001|       2|      2.5|       214100.0|
|  98001|       2|     2.75|       239475.0|
|  98001|       3|     0.75|       363000.0|
|  98001|       3|      1.0|      205182.81|
|  98001|       3|      1.5|       224108.5|
+-------+--------+---------+---------------+
only showing top 10 rows



In [7]:
# Detiene la sesión de Spark
spark.stop()
print("Sesión finalizada correctamente.")

Sesión finalizada correctamente.


In [8]:
!pip install fpdf2

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------------------- -------- 1.8/2.3 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 8.9 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   -------- ------------------------------- 1.6/7.1 MB 8.4 MB/s eta 0:00:01
   -------------------- ------------------- 3.7/7.1 MB 9.1 MB/s eta 0:00:01
   ------------------------------- -------- 5.5/7.1 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 9.1 MB/s  0:00:00

   ---------------------------------------- 0/4 [Pillow]
   ---------------------------------------- 0/4 [Pillow]
   ---------------------------------------- 0/4 [Pillow]
   ---------------------------------------- 0/4 [Pillow]
   ---------------------------------------- 0/4 [Pillow]
   ---------- ----------------------------- 1/4 [fonttools]
   ---------- ----------------------------- 1/4 [fonttools]
   ---

In [11]:
# !pip install fpdf2

from fpdf import FPDF
import datetime
import os

# 1. Definición de los contenidos de las celdas (z1 a z7)
z1 = """import os
import sys

# Elimina las variables de entorno conflictivas ANTES de importar PySpark
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]
if "PYSPARK_SUBMIT_ARGS" in os.environ:
    del os.environ["PYSPARK_SUBMIT_ARGS"]

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, round, count

# Configura la sesión de Spark utilizando los binarios internos
spark = SparkSession.builder \\
    .appName("EBAC_Practica_Spark_Patrick") \\
    .master("local[*]") \\
    .config("spark.driver.bindAddress", "127.0.0.1") \\
    .getOrCreate()

print(f"Sesión de Spark iniciada. Versión: {spark.version}")"""

z2 = """# Carga el archivo CSV a una estructura DataFrame de Spark
ruta_archivo = os.path.join("C:\\\\Users\\\\patri\\\\Downloads", "kc_house_data (1).csv")
df = spark.read.csv(ruta_archivo, header=True, inferSchema=True)

# Diagnostica la estructura y muestra las primeras filas
print("--- Esquema de Datos ---")
df.printSchema()
df.show(5)"""

z3 = """# Limpia los datos: elimina nulos y filtra valores en cero
df_clean = df.na.drop() \\
    .filter((col("bedrooms") > 0) & (col("bathrooms") > 0) & (col("price") > 0))

print(f"Total de registros tras la limpieza: {df_clean.count()}")"""

z4 = """# Ordena el listado completo de columnas por zipcode
df_sorted = df_clean.orderBy(col("zipcode"))

print("--- Listado completo ordenado por zipcode ---")
df_sorted.show(5)"""

z5 = """# Encuentra el zipcode con mayor número de casas
top_zip_row = df_clean.groupBy("zipcode").agg(count("*").alias("total_casas")).orderBy(col("total_casas").desc()).first()
top_zip = top_zip_row["zipcode"]
print(f"Zipcode con mayor número de casas: {top_zip}")

# Filtra por el zipcode seleccionado y calcula el promedio de precio y tamaño en m2
df_top = df_clean.filter(col("zipcode") == top_zip)
df_res = df_top.agg(
    round(avg("price"), 2).alias("precio_promedio"),
    round(avg("sqft_living") * 0.092903, 2).alias("tamaño_m2_promedio")
)

print(f"--- Promedios calculados para el zipcode {top_zip} ---")
df_res.show()"""

z6 = """# Agrupa por zipcode, número de habitaciones y baños para obtener el precio promedio
df_grouped = df_clean.groupBy("zipcode", "bedrooms", "bathrooms").agg(
    round(avg("price"), 2).alias("precio_promedio")
).orderBy("zipcode", "bedrooms", "bathrooms")

# Muestra el resultado final del agrupamiento
print("--- Agrupamiento por zipcode, habitaciones y baños ---")
df_grouped.show(10)"""

z7 = """# Detiene la sesión de Spark
spark.stop()
print("Sesión finalizada correctamente.")"""

# 2. Configuración de la clase del Reporte PDF
class ReporteSpark(FPDF):
    def header(self):
        self.set_font("helvetica", "B", 10)
        self.set_text_color(100)
        self.cell(0, 10, "EBAC - Profesión: Analista de Datos | Práctica Big Data", border=False, ln=True, align="R")
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font("helvetica", "I", 8)
        self.cell(0, 10, f"Página {self.page_no()} | Generado por Patrick Salvador Hernández Arias", align="C")

# 3. Función de construcción del PDF
def crear_pdf_entrega(contenido_celdas, nombre_archivo):
    pdf = ReporteSpark()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Portada
    pdf.add_page()
    pdf.set_font("helvetica", "B", 24)
    pdf.ln(60)
    pdf.cell(0, 20, "Proyecto: Procesamiento con Spark", ln=True, align="C")
    pdf.set_font("helvetica", "", 16)
    pdf.cell(0, 10, "Práctica Módulo 52", ln=True, align="C")
    pdf.ln(10)
    pdf.set_font("helvetica", "I", 12)
    pdf.cell(0, 10, f"Fecha: {datetime.date.today()}", ln=True, align="C")
    
    titulos = [
        "1. Configuración de Plataforma Spark",
        "2. Importación y Diagnóstico",
        "3. Limpieza y Tratamiento de Datos",
        "4. Selección 1: Ordenamiento por Zipcode",
        "5. Selección 2: Análisis de Zipcode de Alta Densidad",
        "6. Agrupamiento Multidimensional",
        "7. Finalización de Recursos"
    ]
    
    for titulo, codigo in zip(titulos, contenido_celdas):
        pdf.add_page()
        pdf.set_font("helvetica", "B", 14)
        pdf.set_text_color(0, 51, 102)
        pdf.cell(0, 10, titulo, ln=True)
        pdf.ln(5)
        
        # Bloque de Código (Manejo estándar de texto en UTF-8)
        pdf.set_font("courier", "", 9)
        pdf.set_text_color(0)
        pdf.set_fill_color(245, 245, 245)
        pdf.multi_cell(0, 5, codigo, border=1, fill=True)
        pdf.ln(10)
        
        pdf.set_font("helvetica", "", 11)
        descripcion = f"En esta etapa se ejecuta la lógica de {titulo.lower()}. Se asegura la integridad de los datos siguiendo los estándares operativos."
        pdf.multi_cell(0, 7, descripcion)

    # Guarda el archivo
    ruta_final = os.path.join("C:\\Users\\patri\\Downloads", f"{nombre_archivo}.pdf")
    pdf.output(ruta_final)
    print(f"Reporte generado exitosamente en: {ruta_final}")

# 4. Ejecución
lista_celdas = [z1, z2, z3, z4, z5, z6, z7]
crear_pdf_entrega(lista_celdas, "Entrega_Spark_Patrick")

Reporte generado exitosamente en: C:\Users\patri\Downloads\Entrega_Spark_Patrick.pdf


C:\Users\patri\AppData\Local\Temp\ipykernel_43528\3138363049.py:83: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, "EBAC - Profesión: Analista de Datos | Práctica Big Data", border=False, ln=True, align="R")
C:\Users\patri\AppData\Local\Temp\ipykernel_43528\3138363049.py:100: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 20, "Proyecto: Procesamiento con Spark", ln=True, align="C")
C:\Users\patri\AppData\Local\Temp\ipykernel_43528\3138363049.py:102: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Práctica Módulo 52", ln=True, align="C")
C:\Users\patri\AppData\Local\Temp\ipykernel_43528\3138363049.py:105: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=X